# FastText Embeddings Training

Train domain-specific FastText embeddings on Steam reviews for sentiment classification.

In [17]:
import pandas as pd
import numpy as np
import re
import fasttext
from pathlib import Path
from tqdm import tqdm

# Paths
BASE_PATH = Path('../steam_data_20251208')
REVIEWS_PATH = BASE_PATH / 'reviews'
OUTPUT_PATH = Path('../models')
OUTPUT_PATH.mkdir(exist_ok=True)

## 1. Load All Reviews

In [18]:
# Load all reviews from all genres
all_reviews = []
genre_folders = [f for f in REVIEWS_PATH.iterdir() if f.is_dir()]

for genre_folder in tqdm(genre_folders, desc='Loading reviews'):
    for csv_file in genre_folder.glob('reviews_*.csv'):
        try:
            df = pd.read_csv(csv_file)
            if 'review_text' in df.columns:
                all_reviews.append(df[['review_text']])
        except Exception as e:
            print(f"Error loading {csv_file.name}: {e}")

reviews_df = pd.concat(all_reviews, ignore_index=True)
print(f"Loaded {len(reviews_df):,} reviews")

Loading reviews: 100%|██████████| 12/12 [00:00<00:00, 23.75it/s]

Loaded 103,946 reviews


## 2. Preprocessing

In [19]:
import sys
sys.path.insert(0, '..')
from preprocessing import preprocess_for_fasttext

# Test preprocessing
sample = "This game is TERRIBLE!!! Don't buy it... https://store.steam.com <br/>"
print(f"Before: {sample}")
print(f"After:  {preprocess_for_fasttext(sample)}")

Before: This game is TERRIBLE!!! Don't buy it... https://store.steam.com <br/>
After:  this game is terrible ! ! ! don ' t buy it . . .


In [20]:
# Apply preprocessing
tqdm.pandas(desc='Preprocessing')
reviews_df['processed'] = reviews_df['review_text'].progress_apply(preprocess_for_fasttext)

# Filter empty reviews
reviews_df = reviews_df[reviews_df['processed'].str.len() > 10]
print(f"Reviews after filtering: {len(reviews_df):,}")

Preprocessing: 100%|██████████| 103946/103946 [00:01<00:00, 52590.72it/s]

Reviews after filtering: 93,099


## 3. Prepare Training File

In [21]:
# Save to text file (one review per line - FastText format)
corpus_path = OUTPUT_PATH / 'fasttext_corpus.txt'

with open(corpus_path, 'w', encoding='utf-8') as f:
    for review in tqdm(reviews_df['processed'], desc='Writing corpus'):
        f.write(review + '\n')

print(f"Corpus saved to {corpus_path}")

Writing corpus: 100%|██████████| 93099/93099 [00:00<00:00, 952766.19it/s]

Corpus saved to ..\models\fasttext_corpus.txt


## 4. Train FastText Model

In [22]:
# Train FastText
model = fasttext.train_unsupervised(
    str(corpus_path),
    model='skipgram',
    dim=100,
    lr=0.05,
    epoch=10,
    minCount=3,
    wordNgrams=2, # Use bigrams
)

print(f"Vocabulary size: {len(model.words):,}")

Vocabulary size: 32,313


## 5. Save Model

In [26]:
# Save model
model_path = OUTPUT_PATH / 'steam_fasttext.bin'
model.save_model(str(model_path))
print(f"Model saved to {model_path}")

Model saved to ..\models\steam_fasttext.bin


## 6. Test Embeddings

In [27]:
# Test word vectors
test_words = ['game', 'fun', 'boring', 'recommend', 'refund', 'dlc', 'grind', 'masterpiece']

print("Most similar words:")
print("=" * 50)
for word in test_words:
    similar = model.get_nearest_neighbors(word, k=5)
    print(f"\n{word}:")
    for score, w in similar:
        print(f"  {w}: {score:.3f}")

Most similar words:

game:
  this: 0.880
  it: 0.820
  .: 0.788
  but: 0.778
  really: 0.750

fun:
  enjoyable: 0.773
  great: 0.744
  entertaining: 0.733
  😎: 0.728
  good: 0.713

boring:
  repetive: 0.812
  repetetive: 0.800
  repetitive: 0.799
  dull: 0.797
  boringly: 0.777

recommend:
  recommmend: 0.944
  reccommend: 0.898
  recommends: 0.897
  recommed: 0.849
  recommened: 0.846

refund:
  refunds: 0.823
  refunded: 0.787
  refunding: 0.782
  requested: 0.709
  2hr: 0.686

dlc:
  dlcs: 0.787
  £12: 0.653
  transmission: 0.636
  $85: 0.631
  expansion: 0.615

grind:
  grind-: 0.895
  grind-y: 0.848
  grindin: 0.840
  grinding: 0.822
  grinds: 0.788

masterpiece:
  masterpieces: 0.917
  masterpice: 0.909
  centerpiece: 0.814
  masterpeace: 0.761
  masterful: 0.659


In [28]:
# Test sentence embedding
sentence = "this game is absolutely amazing"
embedding = model.get_sentence_vector(sentence)
print(f"Sentence: '{sentence}'")
print(f"Embedding shape: {embedding.shape}")
print(f"Embedding (first 10): {embedding[:10]}")

Sentence: 'this game is absolutely amazing'
Embedding shape: (100,)
Embedding (first 10): [ 0.22613528 -0.02735107 -0.10643341 -0.03521904  0.05920029  0.03138863
  0.0205505  -0.01752574  0.00445818  0.10532574]
